# ema-first-moment — faded example 3: Fill the bias-correction divisor

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `ema-first-moment`. Running the beacon reports progress on the `Optimizer: Adam EMA first moment` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam EMA first moment` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`ema-first-moment`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "ema-first-moment"
DD_SUBTOPIC = "Optimizer: Adam EMA first moment"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The zero-initialized first moment is biased low early in training. Adam divides the raw EMA by `1 - beta1**t` at step `t` to debias it, without altering the stored buffer.

## Faded exercise 3

Implement `ema_m_corrected(m, g, beta1, step)`. Update `m` in place with the first-moment recurrence, then return the bias-corrected estimate `m / (1 - beta1**step)`. Complete the blanked correction divisor.

**Fill in:** the bias-correction divisor 1 - beta1**step applied to the raw EMA buffer

In [ ]:
import torch as t

t.manual_seed(5)
beta1 = 0.9

def ema_m_corrected(m, g, beta1, step):
    m.copy_(beta1 * m + (1 - beta1) * g)
    divisor = 1 - beta1 ** step
    return m / divisor

buf = t.zeros(2)
print(ema_m_corrected(buf, t.tensor([1.0, 1.0]), beta1, 1).tolist())


def _test():
    b1 = 0.9
    buf = t.zeros(2)
    g = t.tensor([1.0, 1.0])
    # step 1 from zero: raw m = (1-b1)*g = 0.1; corrected = 0.1 / (1-0.9) = 1.0
    out1 = ema_m_corrected(buf, g, b1, 1)
    assert t.allclose(out1, t.ones(2), atol=1e-6), out1
    # raw buffer must remain the un-corrected EMA value
    assert t.allclose(buf, (1 - b1) * g, atol=1e-6), buf
    # step 2 corrected with same constant grad stays exactly 1.0 (debias property)
    out2 = ema_m_corrected(buf, g, b1, 2)
    assert t.allclose(out2, t.ones(2), atol=1e-6), out2


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(5)
beta1 = 0.9

def ema_m_corrected(m, g, beta1, step):
    m.copy_(beta1 * m + (1 - beta1) * g)
    divisor = 1 - beta1 ** step
    return m / divisor

buf = t.zeros(2)
print(ema_m_corrected(buf, t.tensor([1.0, 1.0]), beta1, 1).tolist())
```
</details>